# 5주차. 4주간의 이야기, GPT-2

In [1]:
%pip install -q torch transformers


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: /Users/kimtaeyeong/miniconda3/envs/nlp-study/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## GPT-2 불러오기

In [2]:
# 직접 조립하는 대신, Hugging Face에 올라온 실제 GPT-2를 그대로 받아온다
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

print(model.config)


/Users/kimtaeyeong/miniconda3/envs/nlp-study/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8918.41it/s]

GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.14.1",
  "use_cache": true,
  "vocab_size": 50257
}



In [3]:
# Table 2의 117M 모델과 설정이 정확히 같은지 확인한다
cfg = model.config
print(f"레이어 수: {cfg.n_layer}   (Table 2: 12)")
print(f"d_model:  {cfg.n_embd}   (Table 2: 768)")
print(f"어휘 크기: {cfg.vocab_size}   (1주차에서 본 그 50,257)")
print(f"컨텍스트 길이: {cfg.n_positions}   (1주차 n_positions)")
print(f"인코더-디코더 사이 Cross-Attention: {cfg.add_cross_attention}")


레이어 수: 12   (Table 2: 12)
d_model:  768   (Table 2: 768)
어휘 크기: 50257   (1주차에서 본 그 50,257)
컨텍스트 길이: 1024   (1주차 n_positions)
인코더-디코더 사이 Cross-Attention: False


## 입력 표현: byte-level BPE

In [4]:
# 문자 카테고리를 넘나드는 병합을 금지했다는 게 실제로 이런 뜻이다
for w in ["dog", "dog.", "dog!", "dog?"]:
    ids = tokenizer.encode(w)
    print(f"{w:6s} -> {tokenizer.convert_ids_to_tokens(ids)}")


dog    -> ['dog']
dog.   -> ['dog', '.']
dog!   -> ['dog', '!']
dog?   -> ['dog', '?']


In [5]:
# 알파벳과 구두점이 따로 떨어져서, "dog"이라는 조각 하나가 네 경우 모두 재사용된다
print("네 단어 모두 첫 토큰이 같은 'dog'인가:",
      all(tokenizer.convert_ids_to_tokens(tokenizer.encode(w))[0] == "dog"
          for w in ["dog", "dog.", "dog!", "dog?"]))


네 단어 모두 첫 토큰이 같은 'dog'인가: True


## Zero-shot: 독해

In [6]:
# fine-tuning 없이, 문서 뒤에 질문과 "A:"만 붙여서 다음 토큰을 그대로 받는다
document = "Tom went to the store to buy some milk. He also bought bread and eggs."
question = "What did Tom buy?"
prompt = f"{document}\nQ: {question}\nA:"

input_ids = tokenizer.encode(prompt, return_tensors="pt")
with torch.no_grad():
    output = model.generate(input_ids, max_new_tokens=10, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)

print(tokenizer.decode(output[0]))


Tom went to the store to buy some milk. He also bought bread and eggs.
Q: What did Tom buy?
A: He bought a couple of eggs.
Q:


## Zero-shot: 번역

In [7]:
# "english = french" 형식의 예시 몇 개를 프롬프트로 주고, 마지막 문장의 이어지는 부분을 번역으로 받는다
prompt = "cat = chat\ndog = chien\nhouse = maison\ngood morning ="

input_ids = tokenizer.encode(prompt, return_tensors="pt")
with torch.no_grad():
    output = model.generate(input_ids, max_new_tokens=6, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)

print(tokenizer.decode(output[0]))


cat = chat
dog = chien
house = maison
good morning = maison
good morning =


## Zero-shot: 요약

In [8]:
# 기사 뒤에 "TL;DR:"만 붙이는 것으로 요약하라는 지시를 대신한다
article = (
    "Scientists have discovered a new species of frog in the Amazon rainforest. "
    "The frog, which is bright orange with blue spots, was found by a team of "
    "researchers during a three-month expedition. Local guides say the frog has "
    "never been seen before in the region."
)
prompt = article + "\nTL;DR:"

input_ids = tokenizer.encode(prompt, return_tensors="pt")
with torch.no_grad():
    output = model.generate(input_ids, max_new_tokens=30, do_sample=True, top_k=2,
                             pad_token_id=tokenizer.eos_token_id)

print(tokenizer.decode(output[0])[len(article):])



TL;DR: The new frog is a new species of frog, and it is a new species of frog. It's not a new frog.

This article


## 모델 크기에 따른 스케일링

In [9]:
# gpt2(117M)와 gpt2-medium(345M)의 perplexity를 같은 문장으로 비교한다
text = (
    "The history of natural language processing began with rule-based systems "
    "in the nineteen fifties. Over the following decades, researchers gradually "
    "moved toward statistical methods, and eventually toward neural networks "
    "that could learn language patterns directly from large amounts of text."
)


def perplexity(model_name):
    m = GPT2LMHeadModel.from_pretrained(model_name)
    m.eval()
    ids = tokenizer.encode(text, return_tensors="pt")
    with torch.no_grad():
        out = m(ids, labels=ids)
    n_params = sum(p.numel() for p in m.parameters())
    return torch.exp(out.loss).item(), n_params


for name in ["gpt2", "gpt2-medium"]:
    ppl, n_params = perplexity(name)
    print(f"{name:12s} | 파라미터 {n_params / 1e6:>4.0f}M | perplexity {ppl:.2f}")



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 12338.15it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


gpt2         | 파라미터  124M | perplexity 29.17



Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 292/292 [00:00<00:00, 12151.49it/s]

gpt2-medium  | 파라미터  355M | perplexity 19.91
